In [1]:
!pip install -q monai nibabel scikit-image plotly scipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 26.5 MB/s eta 0:00:00


In [2]:
import os
import re
import tarfile
import numpy as np
import nibabel as nib
import plotly.graph_objects as go
import plotly.io as pio

from scipy.ndimage import gaussian_filter
from skimage import measure

import ipywidgets as widgets
from IPython.display import display, clear_output

pio.renderers.default = "iframe"

# =====================================================
# DATASET
# =====================================================

TAR_PATH = "/kaggle/input/datasets/dschettler8845/brats-2021-task1/BraTS2021_Training_Data.tar"

WORK_DIR = "/kaggle/working/brats_viewer"

os.makedirs(WORK_DIR, exist_ok=True)

# =====================================================
# PATIENT DISCOVERY
# =====================================================

def get_patient_ids():

    ids = []

    with tarfile.open(TAR_PATH, "r") as tar:

        seen = set()

        for member in tar.getmembers():

            m = re.match(
                r"^\./(BraTS2021_\d{5})/",
                member.name
            )

            if m:

                pid = m.group(1)

                if pid not in seen:

                    seen.add(pid)
                    ids.append(pid)

    return sorted(ids)

PATIENT_IDS = get_patient_ids()

# =====================================================
# EXTRACTION
# =====================================================

def extract_patient(patient_id):

    patient_dir = os.path.join(
        WORK_DIR,
        patient_id
    )

    if os.path.exists(patient_dir):
        return patient_dir

    with tarfile.open(TAR_PATH, "r") as tar:

        members = [
            m
            for m in tar.getmembers()
            if m.name.startswith(f"./{patient_id}/")
        ]

        tar.extractall(
            path=WORK_DIR,
            members=members
        )

    return patient_dir

# =====================================================
# MESH HELPER
# =====================================================

def add_mesh(
    fig,
    volume,
    color,
    name,
    opacity
):

    try:

        verts, faces, _, _ = measure.marching_cubes(
            volume,
            level=0.5,
            step_size=1
        )

        fig.add_trace(

            go.Mesh3d(

                x=verts[:,2],
                y=verts[:,1],
                z=verts[:,0],

                i=faces[:,0],
                j=faces[:,1],
                k=faces[:,2],

                color=color,
                opacity=opacity,

                name=name,
                showlegend=True,

                lighting=dict(
                    ambient=0.4,
                    diffuse=0.8,
                    specular=0.2,
                    roughness=0.5
                ),

                lightposition=dict(
                    x=100,
                    y=200,
                    z=150
                )
            )
        )

    except:
        pass

# =====================================================
# UI
# =====================================================

patient_dropdown = widgets.Dropdown(
    options=PATIENT_IDS,
    description="Patient:"
)

visualization_dropdown = widgets.Dropdown(
    options=[
        "Clinical Tumor Reconstruction",
        "Brain Surface Only",
        "Tumor Only",
        "Necrotic Core Only",
        "Edema Only",
        "Enhancing Tumor Only"
    ],
    description="View:"
)

render_button = widgets.Button(
    description="Render",
    button_style="success"
)

output = widgets.Output()

# =====================================================
# RENDERER
# =====================================================

def render_patient(_):

    with output:

        clear_output(wait=True)

        patient_id = patient_dropdown.value
        view_type = visualization_dropdown.value

        patient_dir = extract_patient(patient_id)

        case_path = os.path.join(
            patient_dir,
            patient_id
        )

        t2 = nib.load(
            case_path + "_t2.nii.gz"
        ).get_fdata()

        seg = nib.load(
            case_path + "_seg.nii.gz"
        ).get_fdata()

        raw_mri = (
            t2 - t2.min()
        ) / (
            t2.max() - t2.min()
        )

        brain = (
            raw_mri > 0.15
        ).astype(np.float32)

        brain = gaussian_filter(
            brain,
            sigma=1
        )

        necrotic = (seg == 1).astype(np.float32)
        edema = (seg == 2).astype(np.float32)
        enhancing = (seg == 4).astype(np.float32)

        fig = go.Figure()

        if view_type == "Clinical Tumor Reconstruction":

            add_mesh(fig, brain, "#FFFFFF", "Brain Surface", 0.10)
            add_mesh(fig, edema, "#00FF4D", "Edema Shell", 0.35)
            add_mesh(fig, necrotic, "#FF004D", "Necrotic Core", 1.0)
            add_mesh(fig, enhancing, "#00D0FF", "Enhancing Core", 1.0)

        elif view_type == "Brain Surface Only":

            add_mesh(fig, brain, "#FFFFFF", "Brain Surface", 0.15)

        elif view_type == "Tumor Only":

            add_mesh(fig, edema, "#00FF4D", "Edema Shell", 0.35)
            add_mesh(fig, necrotic, "#FF004D", "Necrotic Core", 1.0)
            add_mesh(fig, enhancing, "#00D0FF", "Enhancing Core", 1.0)

        elif view_type == "Necrotic Core Only":

            add_mesh(fig, necrotic, "#FF004D", "Necrotic Core", 1.0)

        elif view_type == "Edema Only":

            add_mesh(fig, edema, "#00FF4D", "Edema Shell", 0.5)

        elif view_type == "Enhancing Tumor Only":

            add_mesh(fig, enhancing, "#00D0FF", "Enhancing Core", 1.0)

        fig.update_layout(

            title=f"{patient_id} | {view_type}",

            paper_bgcolor="#050505",
            plot_bgcolor="#050505",

            font=dict(color="white"),

            margin=dict(
                l=0,
                r=0,
                b=0,
                t=40
            ),

            showlegend=True,

            legend=dict(
                bgcolor="rgba(0,0,0,0.4)",
                bordercolor="white",
                borderwidth=1
            ),

            scene=dict(

                xaxis=dict(
                    title="Lateral",
                    showgrid=False,
                    showbackground=False,
                    showticklabels=False
                ),

                yaxis=dict(
                    title="Anterior",
                    showgrid=False,
                    showbackground=False,
                    showticklabels=False
                ),

                zaxis=dict(
                    title="Superior",
                    showgrid=False,
                    showbackground=False,
                    showticklabels=False
                ),

                aspectmode="data",

                camera=dict(
                    eye=dict(
                        x=1.5,
                        y=1.5,
                        z=0.8
                    )
                )
            )
        )

        fig.show()

render_button.on_click(render_patient)

display(
    widgets.HBox([
        patient_dropdown,
        visualization_dropdown,
        render_button
    ])
)

display(output)

Output()